# LeetCode 3505 — Minimum Operations to Make Elements Within K Subarrays Equal bằng Python

- Link: https://leetcode.com/problems/minimum-operations-to-make-elements-within-k-subarrays-equal/
- Độ khó: **Hard**
- Chủ đề: **Trung vị tối thiểu hóa tổng khoảng cách**, **trung vị cửa sổ trượt** (Fenwick tree đếm + tổng), **DP chọn k đoạn không chồng nhau**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minOperations(self, nums: List[int], x: int, k: int) -> int:
```

## 1. Mục tiêu học

1. Biết rằng chi phí nhỏ nhất để làm mọi phần tử của một đoạn bằng nhau (mỗi bước ±1) là `Σ |a − trung vị|`.
2. Tính chi phí đó cho **mọi cửa sổ độ dài `x`** khi trượt: dùng **Fenwick tree** trên giá trị đã nén, lưu cả **số lượng** và **tổng**, tìm phần tử thứ `k` bằng nhảy nhị phân.
3. Viết DP `dp[j][i]` = chi phí nhỏ nhất để có `j` cửa sổ rời nhau trong `i` phần tử đầu.
4. Hiểu vì sao các cửa sổ **không chồng nhau** ⇒ thao tác trên chúng độc lập, tổng chi phí cộng được.
5. Kiểm chứng bằng vét cạn chọn đoạn trên mảng nhỏ.

## 2. Đề bài & ví dụ

Cho mảng `nums` và hai số `x`, `k`. Mỗi thao tác tăng hoặc giảm một phần tử đi 1. Tìm số thao tác ít nhất để có **ít nhất `k` đoạn con rời nhau, mỗi đoạn dài đúng `x`**, trong mỗi đoạn mọi phần tử bằng nhau.

**Ví dụ 1:** `nums = [5,-2,1,3,7,3,6,4,-1], x = 3, k = 2` → `8` (đoạn `[−2,1,3] → [1,1,1]` tốn 5, đoạn `[3,6,4] → [4,4,4]` tốn 3).

**Ví dụ 2:** `nums = [9,-2,-2,-2,1,5], x = 2, k = 2` → `3`.

**Ràng buộc:** `2 ≤ n ≤ 10^5`, `|nums[i]| ≤ 10^6`, `2 ≤ x ≤ n`, `1 ≤ k ≤ 15`, `k·x ≤ n`.

In [1]:
from typing import List
examples = [([5, -2, 1, 3, 7, 3, 6, 4, -1], 3, 2, 8), ([9, -2, -2, -2, 1, 5], 2, 2, 3)]
for nums, x, k, exp in examples:
    print(nums, x, k, '-> kỳ vọng', exp)

[5, -2, 1, 3, 7, 3, 6, 4, -1] 3 2 -> kỳ vọng 8
[9, -2, -2, -2, 1, 5] 2 2 -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1. Trung vị tối thiểu hóa tổng khoảng cách

`f(m) = Σ |a_i − m|` là hàm lồi, nhỏ nhất khi `m` là trung vị (với số phần tử chẵn, mọi giá trị giữa hai phần tử giữa đều được).

In [2]:
w = [-2, 1, 3]
for m in range(-3, 5):
    print(m, sum(abs(a - m) for a in w), '<- trung vị' if m == sorted(w)[1] else '')

-3 11 
-2 8 
-1 7 
0 6 
1 5 <- trung vị
2 6 
3 7 
4 10 


### 3.2. Fenwick tree đếm và tổng

Nén giá trị về chỉ số `0..m−1`. Hai mảng Fenwick: `cnt` (số phần tử) và `sm` (tổng giá trị). Thêm/bớt phần tử `O(log m)`. Tìm phần tử nhỏ thứ `k` bằng nhảy nhị phân trên Fenwick `O(log m)`.

In [3]:
class CountSumBIT:
    def __init__(self, m):
        self.m = m
        self.cnt = [0] * (m + 1)
        self.sm = [0] * (m + 1)
        self.LOG = m.bit_length()

    def add(self, i, c, v):
        i += 1
        while i <= self.m:
            self.cnt[i] += c
            self.sm[i] += v
            i += i & -i

    def prefix(self, i):           # (số lượng, tổng) của các chỉ số 0..i
        i += 1
        c = s = 0
        while i > 0:
            c += self.cnt[i]; s += self.sm[i]
            i -= i & -i
        return c, s

    def kth(self, k):              # chỉ số nén của phần tử nhỏ thứ k (1-based)
        pos, c = 0, 0
        for b in range(self.LOG, -1, -1):
            np_ = pos + (1 << b)
            if np_ <= self.m and c + self.cnt[np_] < k:
                pos = np_
                c += self.cnt[np_]
        return pos

vals = [-2, 1, 3, 7]
bit = CountSumBIT(len(vals))
for v in [-2, 1, 3]:
    bit.add(vals.index(v), 1, v)
print('nhỏ thứ 2 =', vals[bit.kth(2)], '| (số lượng, tổng) <= 1:', bit.prefix(1))

nhỏ thứ 2 = 1 | (số lượng, tổng) <= 1: (2, -1)


### 3.3. Chi phí cửa sổ từ trung vị

Với trung vị `med`, gọi `(c, s)` = số lượng và tổng các phần tử `≤ med` trong cửa sổ, `tot` = tổng cửa sổ:

`cost = med·c − s + (tot − s) − med·(x − c)`.

### 3.4. DP chọn `k` đoạn rời nhau

`dp[j][i]` = chi phí nhỏ nhất để có `j` cửa sổ trong `nums[0..i−1]`:

`dp[j][i] = min(dp[j][i−1], dp[j−1][i−x] + cost[i−x])`.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Bài tách làm hai phần độc lập:

1. **Chi phí từng cửa sổ** `cost[i]` cho cửa sổ bắt đầu tại `i` (độ dài `x`).
2. **Chọn `k` cửa sổ không chồng nhau** với tổng chi phí nhỏ nhất.

Vì các cửa sổ rời nhau, thao tác trên cửa sổ này không ảnh hưởng cửa sổ kia ⇒ tổng chi phí là tổng các `cost`.

Tính tay ví dụ 1 (`x = 3`): cửa sổ `[−2,1,3]` (trung vị 1): `3 + 0 + 2 = 5`; `[3,6,4]` (trung vị 4): `1 + 2 + 0 = 3`. Hai cửa sổ rời nhau (chỉ số 1–3 và 5–7) ⇒ `8`.

In [4]:
def cost_naive(nums, x):
    out = []
    for i in range(len(nums) - x + 1):
        w = sorted(nums[i:i + x])
        med = w[(x - 1) // 2]
        out.append(sum(abs(a - med) for a in w))
    return out

print(cost_naive(examples[0][0], 3))

[7, 5, 6, 4, 4, 3, 7]


## 5. Cách 1 — Brute Force: thử mọi tổ hợp `k` cửa sổ

In [5]:
from functools import lru_cache

def min_ops_brute(nums, x, k):
    cost = cost_naive(nums, x)
    n = len(nums)

    @lru_cache(maxsize=None)
    def go(i, left):                 # đang ở vị trí i, còn phải chọn `left` cửa sổ
        if left == 0:
            return 0
        if i + x * left > n:
            return float('inf')
        best = go(i + 1, left)       # không bắt đầu cửa sổ ở i
        best = min(best, cost[i] + go(i + x, left - 1))
        return best

    return go(0, k)

for nums, x, k, exp in examples:
    print(min_ops_brute(nums, x, k), '| kỳ vọng', exp)

8 | kỳ vọng 8
3 | kỳ vọng 3


Hàm này đã là "DP có nhớ", nhưng chi phí cửa sổ tính thô `O(n · x log x)` ⇒ `10^{10}` khi `x ≈ n/2`.

**Nút thắt:** tính `cost` cho mọi cửa sổ — mỗi cửa sổ sắp xếp lại từ đầu.

## 6. Ý tưởng thuật toán / bước đột phá

**Trung vị cửa sổ trượt:** khi trượt sang phải, chỉ **thêm 1** và **bớt 1** phần tử. Với Fenwick trên giá trị nén, mỗi bước:

- thêm/bớt: `O(log m)`;
- trung vị = phần tử nhỏ thứ `⌈x/2⌉`: `O(log m)`;
- `(c, s)` = số lượng và tổng `≤ med`: `O(log m)` ⇒ `cost` theo công thức 3.3.

Sau đó DP `O(n · k)`.

Quy tắc:

- **Nếu** cửa sổ có chi phí `cost[i]` và kết thúc tại `i + x − 1` **thì** cửa sổ kế tiếp phải bắt đầu từ `i + x`.
- **Nếu** không chọn cửa sổ bắt đầu ở `i` **thì** chuyển sang `i + 1`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Cửa sổ sắp xếp duy trì bằng `bisect` + DP

Giữ cửa sổ dưới dạng list đã sắp (`insort` / xóa bằng `bisect`), lấy trung vị trực tiếp; chi phí tính bằng tổng tiền tố của list đã sắp (`O(x)` mỗi cửa sổ). Nhanh hơn cách 1 nhưng vẫn `O(n · x)`.

In [6]:
from bisect import insort, bisect_left

def window_costs_bisect(nums, x):
    win = sorted(nums[:x])
    out = []
    for i in range(len(nums) - x + 1):
        if i:
            win.pop(bisect_left(win, nums[i - 1]))
            insort(win, nums[i + x - 1])
        med = win[(x - 1) // 2]
        h = (x - 1) // 2 + 1
        out.append(med * h - sum(win[:h]) + sum(win[h:]) - med * (x - h))
    return out

def dp_k_windows(cost, n, x, k):
    INF = float('inf')
    prev = [0] * (n + 1)
    for _ in range(k):
        cur = [INF] * (n + 1)
        for i in range(1, n + 1):
            cur[i] = cur[i - 1]
            if i >= x and prev[i - x] + cost[i - x] < cur[i]:
                cur[i] = prev[i - x] + cost[i - x]
        prev = cur
    return prev[n]

def min_ops_bisect(nums, x, k):
    return dp_k_windows(window_costs_bisect(nums, x), len(nums), x, k)

for nums, x, k, exp in examples:
    print(min_ops_bisect(nums, x, k), '| kỳ vọng', exp)

8 | kỳ vọng 8
3 | kỳ vọng 3


### 7.2. Cách 3 — Fenwick trung vị trượt + DP (lời giải chính)

In [7]:
def window_costs_fenwick(nums, x):
    vals = sorted(set(nums))
    idx = {v: i for i, v in enumerate(vals)}
    bit = CountSumBIT(len(vals))
    out, tot = [], 0
    kth = (x + 1) // 2
    for r, v in enumerate(nums):
        bit.add(idx[v], 1, v)
        tot += v
        if r >= x:
            old = nums[r - x]
            bit.add(idx[old], -1, -old)
            tot -= old
        if r >= x - 1:
            p = bit.kth(kth)
            med = vals[p]
            c, s = bit.prefix(p)
            out.append(med * c - s + (tot - s) - med * (x - c))
    return out

def min_ops_fenwick(nums, x, k):
    return dp_k_windows(window_costs_fenwick(nums, x), len(nums), x, k)

for nums, x, k, exp in examples:
    print(min_ops_fenwick(nums, x, k), '| kỳ vọng', exp)

8 | kỳ vọng 8
3 | kỳ vọng 3


**Độ phức tạp:** `O(n log n)` cho chi phí cửa sổ + `O(n·k)` cho DP.

### 7.3. Cách 4 — DP top-down trên chi phí đã tính (để so sánh)

Cùng chi phí Fenwick nhưng DP đệ quy có nhớ — dễ đọc, nhưng `n·k` trạng thái gọi hàm và độ sâu đệ quy lớn; chỉ minh họa trên dữ liệu vừa.

In [8]:
import sys
sys.setrecursionlimit(10000)

def min_ops_topdown(nums, x, k):
    cost = window_costs_fenwick(nums, x)
    n = len(nums)

    @lru_cache(maxsize=None)
    def go(i, left):
        if left == 0:
            return 0
        if i + x * left > n:
            return float('inf')
        return min(go(i + 1, left), cost[i] + go(i + x, left - 1))

    return go(0, k)

for nums, x, k, exp in examples:
    print(min_ops_topdown(nums, x, k), '| kỳ vọng', exp)

8 | kỳ vọng 8
3 | kỳ vọng 3


## 8. TRACE từng bước

Trace ví dụ 1: in chi phí từng cửa sổ (trung vị và cách tính), rồi bảng DP.

In [9]:
def trace(nums, x, k):
    vals = sorted(set(nums))
    idx = {v: i for i, v in enumerate(vals)}
    bit = CountSumBIT(len(vals))
    tot, costs = 0, []
    for r, v in enumerate(nums):
        bit.add(idx[v], 1, v); tot += v
        if r >= x:
            bit.add(idx[nums[r - x]], -1, -nums[r - x]); tot -= nums[r - x]
        if r >= x - 1:
            p = bit.kth((x + 1) // 2); med = vals[p]; c, s = bit.prefix(p)
            cost = med * c - s + (tot - s) - med * (x - c)
            costs.append(cost)
            print(f"cửa sổ {nums[r - x + 1:r + 1]}: trung vị {med}, (c, s) = ({c}, {s}), tổng {tot} -> chi phí {cost}")
    n, INF = len(nums), float('inf')
    prev = [0] * (n + 1)
    for j in range(1, k + 1):
        cur = [INF] * (n + 1)
        for i in range(1, n + 1):
            cur[i] = cur[i - 1]
            if i >= x:
                cur[i] = min(cur[i], prev[i - x] + costs[i - x])
        print(f"dp[{j}] =", ['∞' if v == INF else v for v in cur])
        prev = cur
    print('đáp án =', prev[n])

trace(*examples[0][:3])

cửa sổ [5, -2, 1]: trung vị 1, (c, s) = (2, -1), tổng 4 -> chi phí 7
cửa sổ [-2, 1, 3]: trung vị 1, (c, s) = (2, -1), tổng 2 -> chi phí 5
cửa sổ [1, 3, 7]: trung vị 3, (c, s) = (2, 4), tổng 11 -> chi phí 6
cửa sổ [3, 7, 3]: trung vị 3, (c, s) = (2, 6), tổng 13 -> chi phí 4
cửa sổ [7, 3, 6]: trung vị 6, (c, s) = (2, 9), tổng 16 -> chi phí 4
cửa sổ [3, 6, 4]: trung vị 4, (c, s) = (2, 7), tổng 13 -> chi phí 3
cửa sổ [6, 4, -1]: trung vị 4, (c, s) = (2, 3), tổng 9 -> chi phí 7
dp[1] = ['∞', '∞', '∞', 7, 5, 5, 4, 4, 3, 3]
dp[2] = ['∞', '∞', '∞', '∞', '∞', '∞', 11, 9, 8, 8]
đáp án = 8


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def minOperations(self, nums: List[int], x: int, k: int) -> int:
        vals = sorted(set(nums))
        idx = {v: i for i, v in enumerate(vals)}
        m = len(vals)
        cnt = [0] * (m + 1)
        sm = [0] * (m + 1)
        LOG = m.bit_length()

        def add(i, c, v):
            i += 1
            while i <= m:
                cnt[i] += c
                sm[i] += v
                i += i & -i

        cost, tot = [], 0
        half = (x + 1) // 2
        for r, v in enumerate(nums):
            add(idx[v], 1, v)
            tot += v
            if r >= x:
                o = nums[r - x]
                add(idx[o], -1, -o)
                tot -= o
            if r >= x - 1:
                pos = c = s = 0
                for b in range(LOG, -1, -1):
                    np_ = pos + (1 << b)
                    if np_ <= m and c + cnt[np_] < half:
                        pos = np_
                        c += cnt[np_]
                        s += sm[np_]
                med = vals[pos]
                # cộng thêm các phần tử bằng med (chỉ số nén pos)
                i = pos + 1
                c2 = s2 = 0
                while i > 0:
                    c2 += cnt[i]; s2 += sm[i]
                    i -= i & -i
                cost.append(med * c2 - s2 + (tot - s2) - med * (x - c2))

        n = len(nums)
        INF = float('inf')
        prev = [0] * (n + 1)
        for _ in range(k):
            cur = [INF] * (n + 1)
            for i in range(1, n + 1):
                best = cur[i - 1]
                if i >= x:
                    cand = prev[i - x] + cost[i - x]
                    if cand < best:
                        best = cand
                cur[i] = best
            prev = cur
        return prev[n]

for nums, x, k, exp in examples:
    got = Solution().minOperations(nums, x, k)
    print(got, 'OK' if got == exp else 'SAI')

8 OK
3 OK


## 10. Vì sao thuật toán đúng

1. **Chi phí một đoạn:** đưa mọi phần tử về cùng giá trị `m` tốn `Σ |a − m|`; hàm lồi theo `m`, nhỏ nhất tại trung vị ⇒ `cost` đúng.
2. **Độc lập giữa các đoạn:** các đoạn rời nhau ⇒ thao tác trên mỗi phần tử chỉ phục vụ một đoạn ⇒ tổng chi phí cộng được. Phần tử ngoài các đoạn không cần sửa.
3. **"Ít nhất k" = "đúng k":** thêm đoạn chỉ tăng chi phí (≥ 0) ⇒ chọn đúng `k`.
4. **DP:** xét `nums[0..i−1]`, hoặc phần tử cuối không thuộc đoạn nào (`dp[j][i−1]`), hoặc thuộc đoạn cuối `[i−x, i−1]` (`dp[j−1][i−x] + cost[i−x]`).
5. **Fenwick:** sau mỗi bước trượt, cấu trúc chứa đúng multiset của cửa sổ; `kth` trả phần tử nhỏ thứ `⌈x/2⌉` (một trung vị); `(c, s)` tính đúng phần "≤ trung vị" kể cả phần tử trùng giá trị.

## 11. Phân tích Time / Space Complexity

| Cách | Chi phí cửa sổ | DP | Space |
|---|---|---|---|
| 1. Sắp xếp từng cửa sổ | `O(n x log x)` | `O(nk)` (có nhớ) | `O(nk)` |
| 2. `bisect` | `O(n x)` | `O(nk)` | `O(n)` |
| 3. Fenwick | `O(n log n)` | `O(nk)` | `O(n)` |
| 4. Fenwick + top-down | `O(n log n)` | `O(nk)` + đệ quy | `O(nk)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'bisect': min_ops_bisect, 'fenwick': min_ops_fenwick, 'topdown': min_ops_topdown,
         'submit': lambda a, x, k: Solution().minOperations(a, x, k)}
edge = [([1, 2], 2, 1, 1), ([3, 3, 3, 3], 2, 2, 0), ([1, 100, 1, 100], 2, 2, 198), ([-10**6, 10**6], 2, 1, 2 * 10**6)]
for nums, x, k, exp in examples + edge:
    assert min_ops_brute(nums, x, k) == exp, (nums, x, k)
    for name, f in impls.items():
        got = f(nums, x, k)
        assert got == exp, (name, nums, x, k, got, exp)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [12]:
import random
random.seed(3505)
cnt = 0
for _ in range(400):
    n = random.randint(2, 14)
    x = random.randint(2, n)
    k = random.randint(1, n // x)
    nums = [random.randint(-6, 6) for _ in range(n)]
    exp = min_ops_brute(nums, x, k)
    assert window_costs_fenwick(nums, x) == cost_naive(nums, x)
    for name, f in impls.items():
        assert f(nums, x, k) == exp, (name, nums, x, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test ngẫu nhiên.')

Khớp vét cạn trên 400 bộ test ngẫu nhiên.


### 12.3. Ca lớn (`n = 10^5`, `k = 15`)

In [13]:
import time
random.seed(5)
n = 10**5
nums = [random.randint(-10**6, 10**6) for _ in range(n)]
for x in (2, 300, 6000):
    t = time.time(); a = Solution().minOperations(nums, x, 15); t1 = time.time() - t
    print(f"x={x:>5}: {a} | {t1:.2f}s")
t = time.time(); b = min_ops_bisect(nums[:20000], 300, 15); t2 = time.time() - t
assert b == Solution().minOperations(nums[:20000], 300, 15)
print(f"bisect (n = 2·10^4, x = 300): {t2:.2f}s")

x=    2: 1295 | 0.89s


x=  300: 2022775559 | 0.99s


x= 6000: 44724078649 | 1.10s


bisect (n = 2·10^4, x = 300): 0.13s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng **trung bình** thay vì trung vị

In [14]:
def cost_mean(w):
    m = round(sum(w) / len(w))
    return sum(abs(a - m) for a in w)

w = [1, 1, 100]
print('trung bình:', cost_mean(w), '| trung vị:', cost_naive(w, 3)[0])

trung bình: 132 | trung vị: 99


**Vì sao sai:** trung bình tối thiểu hóa tổng **bình phương** khoảng cách; với tổng **trị tuyệt đối** cần trung vị.

### Lỗi 2 — Tham lam chọn `k` cửa sổ rẻ nhất (không xét chồng nhau)

In [15]:
def wrong_greedy(nums, x, k):
    cost = cost_naive(nums, x)
    chosen, used = 0, [False] * len(nums)
    total = 0
    for c, i in sorted((c, i) for i, c in enumerate(cost)):
        if chosen == k:
            break
        if not any(used[i:i + x]):
            for j in range(i, i + x):
                used[j] = True
            total += c
            chosen += 1
    return total if chosen == k else float('inf')

random.seed(1)
found = next((nums, x, k) for nums, x, k in
             (([random.randint(-5, 5) for _ in range(8)], 2, 3) for _ in range(5000))
             if wrong_greedy(nums, x, k) != min_ops_brute(nums, x, k))
print('phản ví dụ:', found, '| tham lam:', wrong_greedy(*found), '| đúng:', min_ops_brute(*found))

phản ví dụ: ([-3, 4, -4, -1, -4, 2, 2, 2], 2, 3) | tham lam: 10 | đúng: 9


**Vì sao sai:** chọn cửa sổ rẻ nhất trước có thể "chắn" chỗ của hai cửa sổ khác có tổng rẻ hơn (hoặc làm không đủ chỗ cho `k` cửa sổ).

### Lỗi 3 — Cho phép các cửa sổ **chồng nhau** trong DP (`i − 1` thay vì `i − x`)

In [16]:
def wrong_overlap(nums, x, k):
    cost = cost_naive(nums, x)
    n, INF = len(nums), float('inf')
    prev = [0] * (n + 1)
    for _ in range(k):
        cur = [INF] * (n + 1)
        for i in range(1, n + 1):
            cur[i] = cur[i - 1]
            if i >= x:
                cur[i] = min(cur[i], prev[i - 1] + cost[i - x])     # SAI
        prev = cur
    return prev[n]

nums, x, k, exp = examples[1]
print('sai :', wrong_overlap(nums, x, k), '| đúng:', exp)

sai : 0 | đúng: 3


**Vì sao sai:** cửa sổ trước có thể kết thúc bên trong cửa sổ sau ⇒ các phần tử bị dùng hai lần, chi phí thấp giả tạo.

### Lỗi 4 — Tính chi phí với trung vị nhưng quên phần tử **bằng** trung vị khi tách hai phía

In [17]:
def cost_wrong_ties(w):
    s = sorted(w)
    med = s[(len(s) - 1) // 2]
    low = [a for a in s if a < med]                 # SAI: bỏ các phần tử == med ra khỏi phép đếm
    high = [a for a in s if a > med]
    return med * len(low) - sum(low) + sum(high) - med * (len(s) - len(low))

w = [2, 2, 2, 9]
print('sai :', cost_wrong_ties(w), '| đúng:', cost_naive(w, 4)[0])

sai : 1 | đúng: 7


**Vì sao sai:** công thức `med·c − s + (tot − s) − med·(x − c)` cần `c, s` là số lượng/tổng các phần tử **≤ med** (cả phần tử bằng). Nếu chỉ lấy `< med` thì vế phải bị tính sai.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Làm các phần tử bằng nhau với chi phí ±1" ⇒ **trung vị**.
- "Mọi cửa sổ độ dài cố định" ⇒ **cửa sổ trượt** + cấu trúc thứ tự (Fenwick trên giá trị nén, hai heap, `SortedList`).
- "Chọn `k` đoạn rời nhau, tổng nhỏ nhất" với `k` nhỏ ⇒ DP `O(n·k)`.
- Chi phí cửa sổ cần tổng hai phía trung vị ⇒ Fenwick lưu cả **đếm** và **tổng**.

## 15. Biến thể của bài

1. Đoạn có độ dài **ít nhất** `x` ⇒ DP phức tạp hơn (chi phí đoạn thay đổi), thường cần thêm cấu trúc.
2. Chi phí đổi phần tử là **bình phương** ⇒ dùng trung bình (gần nhất số nguyên) và tổng bình phương tiền tố.
3. Chỉ cần **một** đoạn (`k = 1`) ⇒ min của `cost`.
4. Bài liên quan: LC 480 *Sliding Window Median*, LC 2448 *Minimum Cost to Make Array Equal*, LC 689 *Maximum Sum of 3 Non-Overlapping Subarrays*.

In [18]:
# Biến thể 2: chi phí bình phương, mục tiêu là số nguyên tối ưu gần trung bình
def window_cost_sq(w):
    best = None
    lo, hi = int(sum(w) // len(w)) - 1, int(sum(w) // len(w)) + 2
    for m in range(lo, hi + 1):
        c = sum((a - m) ** 2 for a in w)
        best = c if best is None else min(best, c)
    return best

w = [1, 1, 100]
print('bình phương:', window_cost_sq(w), '| vét cạn:', min(sum((a - m) ** 2 for a in w) for m in range(0, 101)))

bình phương: 6534 | vét cạn: 6534


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1 | Sắp từng cửa sổ + DP | `O(n x log x + nk)` | Oracle |
| 2 | `bisect` + DP | `O(n x + nk)` | `x` nhỏ |
| 3 | Fenwick + DP | `O(n log n + nk)` | **Nộp bài** |
| 4 | Fenwick + DP đệ quy | như 3 | Minh họa |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `minOperations([1, 5, 2, 8], 2, 1)`.

**Bài 2 (tính tay):** `minOperations([1, 5, 2, 8], 2, 2)`.

**Bài 3 (điền TODO):** viết `median_cost(sorted_w)` tính chi phí từ một cửa sổ đã sắp.

**Bài 4 (tự viết lại):** trả về **vị trí bắt đầu** của `k` cửa sổ tối ưu.

**Bài 5 (mini challenge):** cài trung vị trượt bằng **hai heap xóa lười** và đối chiếu với Fenwick.

In [19]:
def median_cost_todo(sorted_w):
    # TODO
    raise NotImplementedError

def chosen_windows_todo(nums, x, k):
    # TODO: lưu lựa chọn trong DP và lần ngược
    raise NotImplementedError

def two_heaps_costs_todo(nums, x):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Chi phí các cửa sổ: `4, 3, 6`. (2) Lấy min. (3) `3`.

**Bài 2.** (1) Hai cửa sổ rời nhau trong 4 phần tử: chỉ có `[1,5]` và `[2,8]`. (2) `4 + 6`. (3) `10`.

**Bài 3.** (1) `med = w[(x−1)//2]`. (2) Tổng hai nửa. (3) So với `cost_naive`.

**Bài 4.** (1) Lưu `take[j][i]`. (2) Lần ngược từ `(k, n)`. (3) Kiểm tra rời nhau và tổng chi phí.

**Bài 5.** (1) Heap max cho nửa dưới, heap min cho nửa trên, cân bằng kích thước. (2) Xóa lười bằng `Counter` các phần tử chờ xóa. (3) Duy trì tổng hai nửa.

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
a1 = Solution().minOperations([1, 5, 2, 8], 2, 1)
a2 = Solution().minOperations([1, 5, 2, 8], 2, 2)
assert a1 == 3 and a2 == 10 == min_ops_brute([1, 5, 2, 8], 2, 2)

def median_cost(sw):
    x = len(sw)
    med = sw[(x - 1) // 2]
    return sum(med - a for a in sw[:(x - 1) // 2 + 1]) + sum(a - med for a in sw[(x - 1) // 2 + 1:])

random.seed(3)
for _ in range(300):
    w = sorted(random.randint(-20, 20) for _ in range(random.randint(1, 9)))
    assert median_cost(w) == min(sum(abs(a - m) for a in w) for m in range(-20, 21))
print('Bài 1:', a1, '| Bài 2:', a2, '| Bài 3: đúng')

Bài 1: 3 | Bài 2: 10 | Bài 3: đúng


In [21]:
# Bài 4: vị trí các cửa sổ tối ưu
def chosen_windows(nums, x, k):
    cost = window_costs_fenwick(nums, x)
    n, INF = len(nums), float('inf')
    dp = [[0] * (n + 1)] + [[INF] * (n + 1) for _ in range(k)]
    take = [[False] * (n + 1) for _ in range(k + 1)]
    for j in range(1, k + 1):
        for i in range(1, n + 1):
            dp[j][i] = dp[j][i - 1]
            if i >= x and dp[j - 1][i - x] + cost[i - x] < dp[j][i]:
                dp[j][i] = dp[j - 1][i - x] + cost[i - x]
                take[j][i] = True
    starts, j, i = [], k, n
    while j:
        if take[j][i]:
            starts.append(i - x); i -= x; j -= 1
        else:
            i -= 1
    return dp[k][n], starts[::-1]

random.seed(4)
for _ in range(200):
    n = random.randint(2, 14); x = random.randint(2, n); k = random.randint(1, n // x)
    nums = [random.randint(-6, 6) for _ in range(n)]
    total, starts = chosen_windows(nums, x, k)
    cost = cost_naive(nums, x)
    assert total == sum(cost[s] for s in starts) == min_ops_brute(nums, x, k)
    assert all(starts[t] + x <= starts[t + 1] for t in range(len(starts) - 1))
print('Bài 4: ví dụ 1 ->', chosen_windows(*examples[0][:3]))

Bài 4: ví dụ 1 -> (8, [1, 5])


In [22]:
# Bài 5: hai heap xóa lười
import heapq
from collections import Counter

def two_heaps_costs(nums, x):
    lo, hi = [], []                 # lo: max-heap (lưu -v), hi: min-heap
    delayed = Counter()
    size_lo = size_hi = 0
    sum_lo = sum_hi = 0

    def prune(h, sign):
        while h and delayed[sign * h[0]]:
            delayed[sign * h[0]] -= 1
            heapq.heappop(h)

    def balance():
        nonlocal size_lo, size_hi, sum_lo, sum_hi
        if size_lo > size_hi + 1:
            v = -heapq.heappop(lo); size_lo -= 1; sum_lo -= v
            heapq.heappush(hi, v); size_hi += 1; sum_hi += v
            prune(lo, -1)
        elif size_lo < size_hi:
            v = heapq.heappop(hi); size_hi -= 1; sum_hi -= v
            heapq.heappush(lo, -v); size_lo += 1; sum_lo += v
            prune(hi, 1)

    def insert(v):
        nonlocal size_lo, size_hi, sum_lo, sum_hi
        if not lo or v <= -lo[0]:
            heapq.heappush(lo, -v); size_lo += 1; sum_lo += v
        else:
            heapq.heappush(hi, v); size_hi += 1; sum_hi += v
        balance()

    def erase(v):
        nonlocal size_lo, size_hi, sum_lo, sum_hi
        delayed[v] += 1
        if v <= -lo[0]:
            size_lo -= 1; sum_lo -= v
            if v == -lo[0]: prune(lo, -1)
        else:
            size_hi -= 1; sum_hi -= v
            if hi and v == hi[0]: prune(hi, 1)
        balance()

    out = []
    for r, v in enumerate(nums):
        insert(v)
        if r >= x:
            erase(nums[r - x])
        if r >= x - 1:
            med = -lo[0]
            out.append(med * size_lo - sum_lo + sum_hi - med * size_hi)
    return out

random.seed(5)
for _ in range(300):
    n = random.randint(2, 20); x = random.randint(2, n)
    nums = [random.randint(-5, 5) for _ in range(n)]
    assert two_heaps_costs(nums, x) == window_costs_fenwick(nums, x), (nums, x)
big = [random.randint(-10**6, 10**6) for _ in range(30000)]
assert two_heaps_costs(big, 777) == window_costs_fenwick(big, 777)
print('Bài 5: hai heap khớp Fenwick')

Bài 5: hai heap khớp Fenwick


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
nén giá trị; Fenwick (cnt, sum); trượt cửa sổ x
med = kth((x+1)//2); (c, s) = prefix(med); cost = med*c - s + (tot - s) - med*(x - c)
dp[0][*] = 0; dp[j][i] = min(dp[j][i-1], dp[j-1][i-x] + cost[i-x]); đáp án dp[k][n]
```

**Checklist:**

- [ ] Dùng **trung vị** (không phải trung bình)?
- [ ] `(c, s)` gồm cả phần tử **bằng** trung vị?
- [ ] DP nhảy `i − x` (không chồng nhau)?
- [ ] `dp[0][*] = 0`, các ô không khả thi = ∞?
- [ ] Chạy thử `n = 10^5`, `k = 15`?